# SMRD - story-held-out evaluation of the neural arms (Colab, T4 GPU)

This notebook needs nothing from earlier runs: it works on any Google account, including one with an empty Drive.

Before running: **Runtime > Change runtime type > T4 GPU > Save**.

The earlier folds only kept near-copies of a tweet together. Here every test fold contains only stories (PHEME: whole
events) that the model never saw in training or validation. Three arms are trained on these folds:

* **lexical**: word counts + logistic regression (CPU).
* **graph**: BiGCN.
* **text_long**: RoBERTa on the source tweet, up to 30 epochs.

The validation fold is kept exactly as before: never trained on, used only for model selection. Each test fold is
predicted once. Results are stored under `checkpoints/cv/story/` on Drive, apart from the earlier results. Finished
folds are skipped if you run a cell again. **Run the last cell before closing the tab or switching account.**

Lexical arm for reference (pooled accuracy, already run on CPU):

| | Twitter16 | Twitter15 | PHEME |
|---|---|---|---|
| earlier (grouped) folds | 0.848 | 0.840 | 0.871 |
| story-held-out folds | 0.683 | 0.686 | 0.701 |

In [ ]:
#@title 1. Dependencies and GPU check
import torch, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPU:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('NO GPU: choose Runtime > Change runtime type > T4 GPU, then run this cell again')
%pip install -q 'transformers==5.22.1' 'tokenizers==0.23.3' 'safetensors==0.8.0'     'huggingface_hub==2.4.0' 'pandas==3.1.0' 'scikit-learn==1.9.1' 'pytest==9.2.0'

In [ ]:
#@title 2. Mount Drive, get the story-eval branch
import os, pathlib
from google.colab import drive
drive.mount('/content/drive')

BRANCH = 'story-eval'
CKPT_ROOT = '/content/drive/MyDrive/smrd/checkpoints/cv'
os.makedirs(CKPT_ROOT, exist_ok=True)

PROJECT_ROOT = pathlib.Path('/content/smrd-reproduction')
if not PROJECT_ROOT.exists():
    !git clone -b {BRANCH} https://github.com/Miyonk/smrd-reproduction.git {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git fetch -q origin && git checkout -q {BRANCH} && git pull -q origin {BRANCH}
os.chdir(PROJECT_ROOT)
!git log --oneline -1

In [ ]:
#@title 3. Unit tests (~3 min) - stop here if anything fails
!python -m pytest tests/test_cv.py tests/test_lexical_stack.py tests/test_encoder_pipeline.py tests/test_propagation.py -q

## twitter16

Run this one first and paste the output of 4d. Step c took about 12 minutes on the grouped folds.

In [ ]:
#@title 4a. twitter16: lexical arm (CPU)
!python -m src.run_cv --split story train --dataset twitter16 --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4b. twitter16: BiGCN (GPU)
!python -m src.run_cv --split story train --dataset twitter16 --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4c. twitter16: RoBERTa, up to 30 epochs (GPU)
!python -m src.run_cv --split story train --dataset twitter16 --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 4d. twitter16: compare the three arms
!python -m src.run_cv --split story compare --dataset twitter16 --ckpt-root {CKPT_ROOT}

## twitter15

About twice as long as Twitter16.

In [ ]:
#@title 5a. twitter15: lexical arm (CPU)
!python -m src.run_cv --split story train --dataset twitter15 --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5b. twitter15: BiGCN (GPU)
!python -m src.run_cv --split story train --dataset twitter15 --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5c. twitter15: RoBERTa, up to 30 epochs (GPU)
!python -m src.run_cv --split story train --dataset twitter15 --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 5d. twitter15: compare the three arms
!python -m src.run_cv --split story compare --dataset twitter15 --ckpt-root {CKPT_ROOT}

## pheme

The most expensive: step c can take an hour or more of GPU. Run it only if I ask for it.

In [ ]:
#@title 6a. pheme: lexical arm (CPU)
!python -m src.run_cv --split story train --dataset pheme --arm lexical --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6b. pheme: BiGCN (GPU)
!python -m src.run_cv --split story train --dataset pheme --arm graph --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6c. pheme: RoBERTa, up to 30 epochs (GPU)
!python -m src.run_cv --split story train --dataset pheme --arm text_long --light --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 6d. pheme: compare the three arms
!python -m src.run_cv --split story compare --dataset pheme --ckpt-root {CKPT_ROOT}

In [ ]:
#@title 7. All comparisons again (saved to Drive)
import glob, shutil
!python -m src.run_cv --split story compare --all-datasets --ckpt-root {CKPT_ROOT}
os.makedirs(CKPT_ROOT + '/story', exist_ok=True)
for p in glob.glob('results/cv/*_story.json'):
    shutil.copy(p, CKPT_ROOT + '/story')

In [ ]:
#@title 8. Flush Drive (run before closing the tab or switching account)
drive.flush_and_unmount()
print('Drive flushed and unmounted')